<a href="https://colab.research.google.com/github/DraconMagnus/IA-Aplicada/blob/main/Carlos_KLein_Copy_of_Copia_de_Hands_On_Fine_Tuning_con_LoRA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **HANDS - ON: FINE-TUNING CON LORA**

Una vez vista la masterclass ***Fine-tuning y Evaluación de Modelos***, se proporciona el siguiente ***Colab*** para ejecutar, en vivo, un fine-tuning real con LoRA sobre un modelo Llama ligero, y medir su mejora con una métrica objetiva.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1cKZ_hCf231RE84FDvGkEiKMH6ZDkVZzH?usp=drive_link).

## **CONFIGURACIÓN DEL ENTORNO**

### **INSTALAR LIBRERÍAS**

En este Tema **no necesitas API key ni token**. El modelo base es una copia pública de **Llama 3.2 de 1B Instruct** y sus pesos se descargan directamente del Hub público de Hugging Face; si Colab muestra un aviso de *"unauthenticated requests to the HF Hub"* es normal, solo indica que descargas sin cuenta.

In [ ]:
# Instalar librerías (no se necesita token ni inicio de sesión)
!pip install transformers peft accelerate trl --quiet

import torch

### **CARGAR EL MODELO BASE**

Usamos un modelo Llama ligero (pocos parámetros) para que el fine-tuning corra en minutos sobre la GPU T4 gratuita de Colab, sin necesitar cuantización adicional.

El modelo base es **Llama 3.2 de 1B Instruct** de Meta (unos 1,000 millones de parámetros). Para no pedirte cuenta, token ni aprobación de licencia, cargamos una copia pública del mismo modelo alojada en Hugging Face por Unsloth; la versión oficial es `meta-llama/Llama-3.2-1B-Instruct`, pero requiere iniciar sesión y aceptar la licencia de Meta.

In [ ]:
# Cargar el modelo base de Llama y su tokenizer
import logging

logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers.utils import logging

logging.set_verbosity_error()

modelo_base = "unsloth/Llama-3.2-1B-Instruct"

# variante oficial de Meta (requiere cuenta de Hugging Face, token y aceptar la licencia): "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token  # Llama 3.2 no define token de relleno

modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")

print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo base cargado: unsloth/Llama-3.2-1B-Instruct


### **ANTES DEL FINE-TUNING: LÍNEA BASE**

Antes de ajustar nada, probamos el modelo base con un prompt de ejemplo para tener un punto de comparación. El modelo aún no conoce el tono ni el formato que le vamos a enseñar.

In [ ]:
# Definir una función para generar texto y probar el modelo base con un prompt de ejemplo
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):

    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)

    salida = modelo_a_usar.generate(

        **entrada,

        max_new_tokens=max_new_tokens,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id,

        no_repeat_ngram_size=3,

    )

    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]

    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)

    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)

print(respuesta_base)

No puedo ayudarte con eso. Si deseas cancelar tu pedido, debes comunicarte con el cliente al cliente o contactar con el servicio de atención al cliente de la tienda.


### **PREPARAR LOS DATOS DE ENTRENAMIENTO**

El fine-tuning necesita ejemplos de entrada y salida que muestren el comportamiento que queremos enseñarle al modelo. Con pocos ejemplos (5 a 10) es suficiente para una demo — no es un dataset de producción.

In [ ]:
# Definir una lista de ejemplos (entrada -> respuesta esperada) y convertirla en dataset
from datasets import Dataset

ejemplos = [

    {"texto": "Cliente: ¿Puedo cambiar mi pedido después de pagarlo?\nAgente: Sí, puedes "

     "solicitar el cambio dentro de la primera hora escribiendo a soporte@tienda.com."},

    {"texto": "Cliente: ¿Cuánto tarda el reembolso?\nAgente: El reembolso se refleja en un plazo de 5 a 7 días hábiles."},

    {"texto": "Cliente: ¿Tienen envío el mismo día?\nAgente: Sí, disponible en zonas seleccionadas si el pedido se confirma antes de las 12:00."},

    {"texto": "Cliente: ¿Puedo pagar en el momento de la entrega?\nAgente: Sí, aceptamos pago contra entrega en efectivo o tarjeta."},

    {"texto": "Cliente: ¿Cómo rastreo mi paquete?\nAgente: Puedes rastrearlo con el número de guía en la sección 'Mis pedidos' de tu cuenta."},

]

dataset = Dataset.from_list(ejemplos)

dataset


Dataset({
    features: ['texto'],
    num_rows: 5
})

## **REALIZAR FINE-TUNING**

### **CONFIGURAR Y APLICAR LORA**

LoRA agrega matrices pequeñas entrenables sin tocar los pesos originales del modelo — por eso es tan ligero comparado con un fine-tuning completo.

In [ ]:
# Configurar LoRA (rango, alpha, módulos objetivo) y aplicarlo al modelo base
!pip uninstall -y torchao > /dev/null 2>&1 # evita un error de peft si Colab trae un torchao viejo

from peft import LoraConfig, get_peft_model

from transformers import set_seed

set_seed(42)

config_lora = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=["q_proj", "v_proj"],

    lora_dropout=0.0,

    task_type="CAUSAL_LM"

)

modelo_lora = get_peft_model(modelo, config_lora)

modelo_lora.print_trainable_parameters()

# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


### **ENTRENAR CON LORA**

Con el dataset y LoRA ya configurados, ejecutamos el entrenamiento. La pérdida (*loss*) que reporta el entrenador es nuestra métrica objetiva: debería bajar a medida que el modelo aprende los ejemplos.

In [ ]:
# Configurar el entrenador (SFTTrainer) y ejecutar el fine-tuning
from trl import SFTTrainer, SFTConfig

config_entrenamiento = SFTConfig(

    output_dir="/content/resultados",

    num_train_epochs=30,

    per_device_train_batch_size=5,

    learning_rate=2e-4,

    logging_steps=1,

    dataset_text_field="texto",

    max_length=128,

    report_to="none",

    fp16=True,   # la T4 no soporta bf16 (TRL lo activa por defecto y falla)

    bf16=False,

)

trainer = SFTTrainer(

    model=modelo_lora,

    train_dataset=dataset,

    args=config_entrenamiento,

)

resultado_entrenamiento = trainer.train()

print("Pérdida final:", resultado_entrenamiento.training_loss)

# Dejar el modelo listo para generar texto: TRL lo deja en modo entrenamiento y sin caché

modelo_lora.eval()

modelo_lora.config.use_cache = True

if hasattr(modelo_lora, "gradient_checkpointing_disable"):

    modelo_lora.gradient_checkpointing_disable()

Adding EOS to train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/5 [00:00<?, ? examples/s]

{'loss': '2.84', 'grad_norm': '1.65', 'learning_rate': '0.0002', 'entropy': '2.811', 'mean_token_accuracy': '0.4845', 'num_tokens': '199', 'epoch': '1'}
{'loss': '2.778', 'grad_norm': '1.6', 'learning_rate': '0.0001933', 'entropy': '2.787', 'mean_token_accuracy': '0.4897', 'num_tokens': '398', 'epoch': '2'}
{'loss': '2.7', 'grad_norm': '1.734', 'learning_rate': '0.0001867', 'entropy': '2.755', 'mean_token_accuracy': '0.5', 'num_tokens': '597', 'epoch': '3'}
{'loss': '2.615', 'grad_norm': '1.836', 'learning_rate': '0.00018', 'entropy': '2.714', 'mean_token_accuracy': '0.5206', 'num_tokens': '796', 'epoch': '4'}
{'loss': '2.524', 'grad_norm': '1.944', 'learning_rate': '0.0001733', 'entropy': '2.666', 'mean_token_accuracy': '0.5309', 'num_tokens': '995', 'epoch': '5'}
{'loss': '2.428', 'grad_norm': '2.164', 'learning_rate': '0.0001667', 'entropy': '2.611', 'mean_token_accuracy': '0.5309', 'num_tokens': '1194', 'epoch': '6'}
{'loss': '2.325', 'grad_norm': '2.421', 'learning_rate': '0.00016

### **DESPUÉS DEL FINE-TUNING: MEDIR LA MEJORA**

Compararemos la pérdida antes y después del entrenamiento como métrica objetiva.



In [ ]:
# Comparar la pérdidas
perdida_inicial = trainer.state.log_history[0]['loss']

perdida_final = resultado_entrenamiento.training_loss

print(f"Pérdida al inicio del entrenamiento: {perdida_inicial:.2f}")

print(f"Pérdida final del entrenamiento: {perdida_final:.2f}")

print(f"Reducción: {(1 - perdida_final/perdida_inicial) * 100:.0f}%")

# trainer.state.log_history[0]['loss'] es la pérdida después del primer paso registrado,

# no la pérdida real del modelo sin ningún entrenamiento.

Pérdida al inicio del entrenamiento: 2.84
Pérdida final del entrenamiento: 1.78
Reducción: 37%


### **REFERENCIA CON UN MODELO MÁS GRANDE (OLLAMA)**

Para tener un punto de comparación, instalamos Ollama (igual que en el Tema 1) y le enviamos el mismo prompt a **Llama 3.1 de 8B**, un modelo con unas 8 veces más parámetros que el modelo que acabamos de ajustar. Si tu sesión de Colab es nueva, esto descarga ~5 GB.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1

!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")

Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess

import time

import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):

    try:

        requests.get("http://localhost:11434")

        print("Servidor de Ollama listo.")

        break

    except requests.exceptions.ConnectionError:

        time.sleep(1)

else:

    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.12


In [ ]:
# Descargar el modelo Llama (la primera vez tarda unos minutos)
MODELO = "llama3.1:8b"

!ollama pull {MODELO}

!ollama list


NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    


In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

client = ollama.Client(host="http://localhost:11434")

print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


Además de la respuesta de Llama 3.2 1B ajustado, se muestra la respuesta de Llama 3.1 8B (vía Ollama) para el mismo prompt, como punto de comparación con un modelo con muchos más parámetros y datos de entrenamiento. Con solo 5 ejemplos, el modelo de 1B aprende el tono y el formato de las respuestas, pero también puede inventar detalles que no están en los datos.

In [ ]:
# Referencia cualitativa: Llama 3.2 1B + LoRA vs. un Llama más grande (Llama 3.1 8B en Ollama)
import gc

gc.collect()

torch.cuda.empty_cache()  # libera memoria de la GPU antes de cargar el modelo grande

respuesta_ajustada = generar_respuesta(modelo_lora, prompt_prueba)

print("Respuesta del modelo ajustado (Llama 3.2 1B + LoRA):\n", respuesta_ajustada)

response = client.chat(

    model=MODELO,

    messages=[{"role": "user", "content": prompt_prueba + " Respuesta muy breve y corta."}]

)

print("\nRespuesta de referencia (Llama 3.1 8B / Ollama):\n", response.message.content)


Respuesta del modelo ajustado (Llama 3.2 1B + LoRA):
 Sí, puedes cambiar tu pedido dentro de las 24 horas.

Respuesta de referencia (Llama 3.1 8B / Ollama):
 Lo siento, no podemos hacer cambios después de un pago.


## **CHALLENGE: AJUSTE DE TONO CON LORA**

Una vez visto el ***Hands-On: Fine-tuning con LoRA***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se ajustará un modelo Llama ligero con un dataset propio para enseñarle un tono o formato de respuesta específico, comparando la pérdida **antes** y **después** del fine-tuning como métrica objetiva.

**IMPORTANTE:** Para su revisión, es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.

### **INSTRUCCIONES:**

**1. Carga el modelo y define tu dataset:**

   * Instala las librerías y carga el modelo base junto con la función `generar_respuesta`.

   * Construye una lista llamada `ejemplos` con al menos 4 pares de entrada/respuesta que reflejen el tono o formato que quieres enseñarle al modelo, y conviértela en `dataset`.

In [1]:
# Instalar librerías (no se necesita token ni inicio de sesión)
!pip install transformers peft accelerate trl --quiet

import torch

In [2]:
# Cargar el modelo base de Llama y su tokenizer
import logging

logging.getLogger("huggingface_hub.utils._http").setLevel(logging.ERROR)

from transformers import AutoModelForCausalLM, AutoTokenizer

from transformers.utils import logging

logging.set_verbosity_error()

modelo_base = "unsloth/Llama-3.2-1B-Instruct"

# variante oficial de Meta (requiere cuenta de Hugging Face, token y aceptar la licencia): "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(modelo_base)

if tokenizer.pad_token is None:

    tokenizer.pad_token = tokenizer.eos_token  # Llama 3.2 no define token de relleno

modelo = AutoModelForCausalLM.from_pretrained(modelo_base, dtype=torch.float16, device_map="auto")

print("Modelo base cargado:", modelo_base)

config.json:   0%|          | 0.00/894 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/54.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/3.83k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.47GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Modelo base cargado: unsloth/Llama-3.2-1B-Instruct


In [3]:
# Definir la funcion de generacion de texto
def generar_respuesta(modelo_a_usar, prompt, max_new_tokens=60):

    entrada = tokenizer(prompt, return_tensors="pt").to(modelo_a_usar.device)

    salida = modelo_a_usar.generate(

        **entrada,

        max_new_tokens=max_new_tokens,

        do_sample=False,

        pad_token_id=tokenizer.eos_token_id,

        no_repeat_ngram_size=3,

    )

    tokens_nuevos = salida[0][entrada["input_ids"].shape[1]:]

    texto_generado = tokenizer.decode(tokens_nuevos, skip_special_tokens=True)

    return texto_generado.split("\n")[0].strip()

prompt_prueba = "Cliente: ¿Cuáles son las condiciones de compra de los productos eletrónicos?\nAgente:"

respuesta_base = generar_respuesta(modelo, prompt_prueba)

print(respuesta_base)

Estoy aquí para ayudarte con tu compra de productos eletronicos. ¿Podrías proporcionarme más información sobre las condiciones que deseas cumplir?


In [4]:
# Definir la lista ejemplos y convertirla en dataset
from datasets import Dataset

ejemplos = [
    {"texto": "Cliente: ¿Puedo comprar sin crear una cuenta?\nAgente: Sí, selecciona la opción 'Comprar como invitado' al finalizar tu compra."},

    {"texto": "Cliente: ¿Cómo solicito una factura?\nAgente: Ingresa a la sección 'Facturación' y proporciona tu número de pedido y tus datos fiscales."},

    {"texto": "Cliente: ¿Qué hago si recibí un artículo equivocado?\nAgente: Escríbenos a ayuda@tiendaejemplo.com con tu número de pedido y una fotografía del artículo para gestionar el reemplazo."},

    {"texto": "Cliente: ¿Los precios incluyen impuestos?\nAgente: Sí, todos los precios publicados incluyen impuestos. El costo de envío se muestra antes de confirmar la compra."},

    {"texto": "Cliente: ¿Cómo aplico un cupón de descuento?\nAgente: Introduce el código en el campo 'Cupón' del carrito y pulsa 'Aplicar' antes de realizar el pago."},

    {"texto": "Cliente: ¿Puedo recoger mi compra en una sucursal?\nAgente: Sí, selecciona 'Recoger en tienda' y elige una sucursal disponible. Te avisaremos cuando tu compra esté lista."},

    {"texto": "Cliente: ¿Qué garantía tienen los productos?\nAgente: Los productos cuentan con una garantía de 12 meses por defectos de fabricación. Conserva tu comprobante de compra."},

    {"texto": "Cliente: ¿Cómo sé si volverán a tener un producto agotado?\nAgente: Abre la página del producto y selecciona 'Avisarme cuando esté disponible' para recibir una notificación por correo."},

    {"texto": "Cliente: ¿Venden tarjetas de regalo?\nAgente: Sí, puedes adquirir tarjetas de regalo digitales en la sección 'Regalos' y enviarlas al correo de la persona que elijas."},

    {"texto": "Cliente: ¿Qué hago si mi cupón aparece como inválido?\nAgente: Revisa su fecha de vencimiento y las condiciones de uso. Si cumple los requisitos y el problema continúa, contacta a soporte."}
]

dataset = Dataset.from_list(ejemplos)

dataset

Dataset({
    features: ['texto'],
    num_rows: 10
})

**2. Prueba el modelo base:** Genera una respuesta con el modelo sin ajustar para un prompt de prueba y guárdala en `respuesta_base`.

In [5]:
# Probar el modelo base con un prompt de prueba y guardar el resultado en respuesta_base
# Probar el modelo antes del ajuste
prompt_prueba = (
    "Cliente: ¿Qué debo hacer para obtener una factura de mi compra?\nAgente:"
)

modelo.eval()

with torch.no_grad():
    respuesta_base = generar_respuesta(modelo, prompt_prueba)

print("Respuesta del modelo base:")
print(respuesta_base)

Respuesta del modelo base:
Bueno, para obtener la factura de tu compra, puedes seguir estos pasos:


**3. Configura y aplica LoRA:** Fija una semilla con `set_seed` y define tu `LoraConfig` (rango, alpha, módulos objetivo, dropout en 0) y aplícalo al modelo base.

In [7]:
# Configurar LoraConfig y aplicarlo al modelo base
!pip uninstall -y torchao > /dev/null 2>&1 # evita un error de peft si Colab trae un torchao viejo

from peft import LoraConfig, get_peft_model

from transformers import set_seed

set_seed(42)

config_lora = LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=["q_proj", "v_proj"],

    lora_dropout=0.0,

    task_type="CAUSAL_LM"

)

modelo_lora = get_peft_model(modelo, config_lora)

modelo_lora.print_trainable_parameters()

# r más alto = más capacidad para aprender, pero también más parámetros entrenables

trainable params: 851,968 || all params: 1,236,666,368 || trainable%: 0.0689


**4. Entrena:** Configura el `SFTTrainer` con tu dataset y ejecuta el fine-tuning; guarda la pérdida final en `perdida_final`.

In [8]:
# Configurar el Trainer con el dataset y ejecutar el fine-tuning; guardar la pérdida final en perdida_final
from trl import SFTConfig, SFTTrainer

tokenizer.padding_side = "right"

config_entrenamiento = SFTConfig(
    output_dir="./resultados_lora",
    dataset_text_field="texto",
    max_length=256,
    packing=False,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,
    num_train_epochs=10,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=False,
    bf16=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="adamw_torch",
    seed=42
)

trainer = SFTTrainer(
    model=modelo_lora,
    args=config_entrenamiento,
    train_dataset=dataset,
    eval_dataset=dataset,
    processing_class=tokenizer
)

# Evaluar antes y después sobre los mismos ejemplos
perdida_inicial = trainer.evaluate()["eval_loss"]

print(f"Pérdida inicial: {perdida_inicial:.4f}")

resultado_entrenamiento = trainer.train()

perdida_final = trainer.evaluate()["eval_loss"]

print(f"Pérdida final: {perdida_final:.4f}")

Adding EOS to train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/10 [00:00<?, ? examples/s]

{'eval_loss': '3.047', 'eval_model_preparation_time': '0.0104', 'eval_runtime': '9.565', 'eval_samples_per_second': '1.045', 'eval_steps_per_second': '1.045', 'eval_entropy': '2.756', 'eval_mean_token_accuracy': '0.4546', 'eval_num_tokens': '0', 'epoch': 0}
Pérdida inicial: 3.0471
{'loss': '3.296', 'grad_norm': '2.371', 'learning_rate': '0.0002', 'entropy': '2.768', 'mean_token_accuracy': '0.4082', 'num_tokens': '94', 'epoch': '0.2'}
{'loss': '3.032', 'grad_norm': '1.631', 'learning_rate': '0.000196', 'entropy': '2.663', 'mean_token_accuracy': '0.4669', 'num_tokens': '186', 'epoch': '0.4'}
{'loss': '2.728', 'grad_norm': '1.646', 'learning_rate': '0.000192', 'entropy': '2.626', 'mean_token_accuracy': '0.4732', 'num_tokens': '283', 'epoch': '0.6'}
{'loss': '2.807', 'grad_norm': '1.919', 'learning_rate': '0.000188', 'entropy': '2.825', 'mean_token_accuracy': '0.5053', 'num_tokens': '373', 'epoch': '0.8'}
{'loss': '3.095', 'grad_norm': '2.44', 'learning_rate': '0.000184', 'entropy': '2.841

**5. Compara y concluye:** Calcula la reducción entre la pérdida inicial y `perdida_final` como métrica objetiva, y usa la respuesta generada por el modelo ya ajustado solo como referencia cualitativa.

In [9]:
# Comparar la perdida inicial y final del entrenamiento como metrica objetiva
reduccion = perdida_inicial - perdida_final
reduccion_porcentual = (reduccion / perdida_inicial) * 100

print(f"Pérdida inicial: {perdida_inicial:.4f}")
print(f"Pérdida final: {perdida_final:.4f}")
print(f"Reducción absoluta: {reduccion:.4f}")
print(f"Reducción porcentual: {reduccion_porcentual:.2f}%")

if perdida_final < perdida_inicial:
    print(
        "\nConclusión: el ajuste redujo la pérdida sobre los ejemplos "
        "de entrenamiento, lo que indica que el modelo aprendió "
        "a reproducir mejor esos textos."
    )
else:
    print(
        "\nConclusión: el ajuste no redujo la pérdida "
        "con esta configuración."
    )

print(
    "Esta comparación no demuestra generalización: "
    "se evaluaron los mismos datos utilizados para entrenar."
)

Pérdida inicial: 3.0471
Pérdida final: 1.4099
Reducción absoluta: 1.6372
Reducción porcentual: 53.73%

Conclusión: el ajuste redujo la pérdida sobre los ejemplos de entrenamiento, lo que indica que el modelo aprendió a reproducir mejor esos textos.
Esta comparación no demuestra generalización: se evaluaron los mismos datos utilizados para entrenar.


In [10]:
# Como referencia cualitativa (puede variar de sesión a sesión con un dataset tan chico)
modelo_lora.eval()
modelo_lora.config.use_cache = True

with torch.no_grad():
    respuesta_ajustada = generar_respuesta(
        modelo_lora,
        prompt_prueba
    )

print("PROMPT:")
print(prompt_prueba)

print("\nANTES DEL AJUSTE:")
print(respuesta_base)

print("\nDESPUÉS DEL AJUSTE:")
print(respuesta_ajustada)

PROMPT:
Cliente: ¿Qué debo hacer para obtener una factura de mi compra?
Agente:

ANTES DEL AJUSTE:
Bueno, para obtener la factura de tu compra, puedes seguir estos pasos:

DESPUÉS DEL AJUSTE:
Puedes verificar la factura en la sección 'Mis pedidos' o en la pestaña 'Facturas' de tu cuenta.
